# Users Silver Pipeline

In [0]:
import sys
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(repo_root)

print(f"[INFO] Repo root added to path: {repo_root}")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, to_timestamp
from delta.tables import DeltaTable
from functools import reduce

from common.helpers import get_bronze

spark = SparkSession.builder.getOrCreate()

try:
    dbutils.widgets.text("batch_id", "")
    batch_id = dbutils.widgets.get("batch_id")
except:
    batch_id = None

print("[INFO] Products Silver Pipeline Started")

## Load Bronze Data

In [0]:
# Databricks Volume path (Bronze)
users_bronze = "/Volumes/datalake_catalog/datalake_schema/bronze/users"

df_bronze_users = (
    get_bronze(users_bronze, spark=spark)
    .drop("dw_ingested_at", "dw_source_file", "dw_batch_id", "batch_id", "source_table")
)

df_bronze_users.show(2, truncate=False)

## Remove duplicates (user_id + created_at)

In [0]:
window_spec = (
    Window
    .partitionBy("user_id", "created_at")
    .orderBy(F.col("ingest_time").desc())
)

df_with_rn = df_bronze_users.withColumn("rn", row_number().over(window_spec))

df1 = df_with_rn.filter(F.col("rn") == 1).drop("rn")
df1_quarantine = df_with_rn.filter(F.col("rn") > 1).drop("rn")

print("Valid:", df1.count())
print("Quarantine:", df1_quarantine.count())

## Schema Enforcement

In [0]:
df1 = (
    df1
    .withColumn("user_id", col("user_id").cast("int"))
    .withColumn("first_name", col("first_name").cast("string"))
    .withColumn("last_name", col("last_name").cast("string"))
    .withColumn("email", col("email").cast("string"))
    .withColumn("country", col("country").cast("string"))
    .withColumn("age", col("age").cast("int"))
    .withColumn("gender", col("gender").cast("string"))
    .withColumn("acquisition_channel", col("acquisition_channel").cast("string"))
    .withColumn("is_enterprise", col("is_enterprise").cast("boolean"))
    .withColumn("created_at", to_timestamp("created_at"))
    .withColumn("ingest_time", to_timestamp("ingest_time"))
)

## Clean & Standardize

In [0]:
df2 = (
    df1
    .withColumn("first_name", F.trim(col("first_name")))
    .withColumn("last_name", F.trim(col("last_name")))
    .withColumn("email", F.lower(F.trim(col("email"))))
    .withColumn("gender", F.lower(F.trim(col("gender"))))
    .withColumn("acquisition_channel", F.lower(F.trim(col("acquisition_channel"))))
    .withColumn("country", F.initcap(F.trim(col("country"))))
)

## Required field validation

In [0]:
df3 = df2.filter(
    col("user_id").isNotNull() & col("created_at").isNotNull()
)

df3_quarantine = df2.filter(
    col("user_id").isNull() | col("created_at").isNull()
)

## Country Standardization

In [0]:
predefined_countries = [
    "Australia","Brazil","Canada","China","Denmark","France","Germany",
    "India","Indonesia","Italy","Japan","Netherlands","Norway","Philippines",
    "Singapore","South Korea","Spain","Sweden","Thailand","United Kingdom",
    "United States","Vietnam"
]

df4 = (
    df3
    .withColumn("is_valid_country", col("country").isin(predefined_countries))
    .withColumn(
        "country",
        F.when(col("country").isin(predefined_countries), col("country"))
        .otherwise("Other")
    )
)

## Age validation

In [0]:
df5 = df4.filter(col("age").between(18, 100))
df5_quarantine = df4.filter(~col("age").between(18, 100) | col("age").isNull())

## Gender validation

In [0]:
allowed_genders = ["female", "male", "other"]

df6 = df5.filter(col("gender").isin(allowed_genders))
df6_quarantine = df5.filter(~col("gender").isin(allowed_genders) | col("gender").isNull())

## Acquisition channel validation

In [0]:
allowed_channels = ["organic", "paid", "referral"]

df7 = df6.filter(col("acquisition_channel").isin(allowed_channels))
df7_quarantine = df6.filter(~col("acquisition_channel").isin(allowed_channels))

## is_enterprise validation

In [0]:
df8 = df7.filter(col("is_enterprise").isNotNull())
df8_quarantine = df7.filter(col("is_enterprise").isNull())

## Merge all quarantine data

In [0]:
quarantine_dfs = [
    df1_quarantine,
    df3_quarantine,
    df5_quarantine,
    df6_quarantine,
    df7_quarantine,
    df8_quarantine
]

df_quarantine_all = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=True),
    quarantine_dfs
)

## Upsert into Silver Delta Table

In [0]:
silver_path = "/Volumes/datalake_catalog/datalake_schema/silver/users"

df_final = df8

w = Window.partitionBy("user_id").orderBy(col("created_at").desc(), col("ingest_time").desc())

df_final = (
    df_final
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)

if DeltaTable.isDeltaTable(spark, silver_path):
    delta = DeltaTable.forPath(spark, silver_path)
    
    delta.alias("t").merge(
        df_final.alias("s"),
        "t.user_id = s.user_id"
    ).whenMatchedUpdate(
        condition="s.ingest_time > t.ingest_time",
        set={c: f"s.{c}" for c in df_final.columns}
    ).whenNotMatchedInsert(
        values={c: f"s.{c}" for c in df_final.columns}
    ).execute()
else:
    df_final.write.format("delta").mode("overwrite").save(silver_path)